Copyright (C) 2026 Giulio Barni, Eric Madge  
This file is part of `HydróDex`.

`HydróDex` is free software: you can redistribute it and/or modify it under the terms of the GNU General Public License as published by the Free Software Foundation, either version 3 of the License, or (at your option) any later version.

`HydróDex` is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY; without even the implied warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE. See the GNU General Public License for more details.

You should have received a copy of the GNU General Public License along with `HydróDex`. If not, see <https://www.gnu.org/licenses/>. 

# `HydróDex` tutorial

This notebook gives a compact introduction to the public Python interface of `HydróDex`.  The code implements stationary, self-similar hydrodynamics for a constant-speed-of-sound (CSS) equation of state with independent sound speeds $c_+$ and $c_-$, together with the corresponding sound-shell gravitational-wave calculation.

The tutorial is deliberately linear.  It starts from the four physical inputs

$$
(\xi_w,\alpha_N,c_+,c_-),
$$

constructs the fluid profile and its thermodynamic completion, maps the hydrodynamic phase space, and finally propagates one profile through the sound-shell pipeline.

Throughout, $\xi=r/t$ is the self-similar coordinate, $\alpha_N\equiv\alpha_{\bar\theta}(T_N)$ is the pseudo-trace strength in the unperturbed plasma, and $\alpha_+\equiv\alpha_{\bar\theta}(T_+)$ is the corresponding local strength immediately in front of the wall.

# 1. Code structure and main points

The Python module is a thin interface to the compiled C++ implementation.  For most applications only three objects are needed:

- `hydrodex.FluidProfile`: high-level construction of a completed stationary profile from $(\xi_w,\alpha_N,c_+,c_-)$.  It performs the branch classification, wall matching, self-similar evolution, and thermodynamic reconstruction.
- `hydrodex.profile_calculator`: low-level hydrodynamic functions.  It exposes branch boundaries, $\alpha_+\leftrightarrow\alpha_N$ projection, explicit branch construction, and profile utilities.
- `hydrodex.SoundShellSpectrum`: sound-shell observables built from a `FluidProfile`, including the Fourier amplitudes, kinetic spectrum, source correlator, and gravitational-wave spectrum.

The automatic interface is the appropriate starting point when the map to $(\xi_w,\alpha_N)$ is single-valued.  In sufficiently hierarchical regions of the $(c_+,c_-)$ plane the projection can become non-injective.  The optional diagnostic introduced below warns when the chosen sound speeds enter that regime; branch-specific functions remain available when an explicit stationary completion is required.

## 1.1 Setup

The first cell loads the `HydróDex` library.  Plot labels use LaTeX whenever a local LaTeX installation is available; otherwise Matplotlib's mathtext provides a portable fallback.

In [ ]:
import shutil

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

import hydrodex

pc = hydrodex.profile_calculator
ssm = hydrodex.sound_shell_model

use_tex = shutil.which("latex") is not None
plt.rcParams.update({
    "text.usetex": use_tex,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman", "DejaVu Serif"],
    "font.size": 12,
    "axes.labelsize": 13,
    "axes.titlesize": 13,
    "legend.fontsize": 10,
    "figure.dpi": 120,
})

print(f"found hydrodex version : {hydrodex.__version__}")
print(f"hydrodex located in    : {hydrodex.__file__}")
print(f"LaTeX plots            : {use_tex}")

## 1.2 Runtime checks and numerical defaults

`CheckMode` controls how internal checks are handled.  The tutorial uses `WARNING` as the default, i.e. the calculation continues, but non-critical issues remain visible.  Non-critical consistency checks are controlled by `consistency_check`, while the non-injectivity sound-speed check depends on `noninjectivity_check`.  Both can be set and retrieved with the corresponding `set_` and `get_` functions.

The profile step size controls the self-similar ODE resolution.  The value below is conservative for the examples in this notebook.  Production scans should be checked for convergence.

In [ ]:
hydrodex.set_debug(False)
hydrodex.set_consistency_check(hydrodex.CheckMode.WARNING)
hydrodex.set_noninjectivity_check(hydrodex.CheckMode.WARNING)

pc.set_profile_step_size(1.0e-5)

print("consistency check mode:     ", hydrodex.get_consistency_check())
print("non-injectivity check mode: ", hydrodex.get_noninjectivity_check())

## 1.3 Minimal workflow

The high-level hydrodynamic call is simply

```python
profile = hydrodex.FluidProfile(xi_w=xi_w, alpha_N=alpha_N, cp=c_plus, cm=c_minus)
```

Internally, the solver determines the hydrodynamic branch, solves the wall matching conditions, completes the self-similar profile with shocks and/or continuous waves, and reconstructs $w(\xi)$, $e(\xi)$ and $p(\xi)$.  Temperature and entropy require one additional thermodynamic normalization, introduced below.

The CSS equation of state is

$$
e_\pm(T)=a_\pm T^{q_\pm}+\epsilon_\pm,
\qquad
p_\pm(T)=c_\pm^2 a_\pm T^{q_\pm}-\epsilon_\pm,
\qquad
q_\pm=1+\frac{1}{c_\pm^2}.
$$

The automatic physical input is $\alpha_N$, not $\alpha_+$.  When the plasma in front of the wall is perturbed, the two are related by

$$
\alpha_N=\alpha_0+r_w(\alpha_+-\alpha_0),
\qquad
r_w\equiv\frac{w_+}{w_N},
$$

so the projection depends on the completed profile ahead of the wall.

# 2. Hydrodynamics

We use the benchmark

$$
c_+=0.75,\qquad c_-=0.40,
$$

for which all direct, inverse, and mixed solutions exist but do not overlap.  As an example we choose a mixed hybrid profile with $\xi_w = 0.65$ and $\alpha_N = 0.05$.

## 2.1 Physical inputs and non-injectivity

We check that, for the chosen values of $c_+$ and $c_-$, each input $(\xi_w, \alpha_N)$ can correspond to (at most) one valid profile without any degeneracies.  This is checked based on a fitted curve for the critical sound speed $c_{-,{\rm crit}}^{\rm ID^{+}}$ (`inverse_detonation_plus_fold_critical_cm`).  If `check_noninjectivity` returns `True`, *some* part of the hydrodynamic map can become non-injective for the chosen $(c_+,c_-)$ (this does not mean that every point $(\xi_w,\alpha_N)$ is degenerate).

In [ ]:
c_plus = 0.75
c_minus = 0.40
xi_w = 0.65
alpha_N = 0.05

cm_crit = pc.inverse_detonation_plus_fold_critical_cm(c_plus)
noninjective = pc.check_noninjectivity(c_plus, c_minus)

print(f"c_+                  = {c_plus:.6f}")
print(f"c_-                  = {c_minus:.6f}")
print(f"c_-^crit(c_+)        = {cm_crit:.6f}")
print(f"non-injective region = {noninjective}")
print(f"xi_w                 = {xi_w:.6f}")
print(f"alpha_N              = {alpha_N:.6f}")

## 2.2 Construct one completed fluid profile

`FluidProfile` is the primary user interface for calculating the fluid profile.  The wall-frame velocities $v_+$ and $v_-$ are printed here only for diagnostics.  The cosmic-frame profile $v(\xi)$ is what enters the energy budget and the sound-shell calculation.

In [ ]:
profile = hydrodex.FluidProfile(
    xi_w=xi_w,
    alpha_N=alpha_N,
    cp=c_plus,
    cm=c_minus,
    step_size=1.0e-5,
)

sol_type = profile.get_solution_type()
alpha_plus = profile.get_wall_transition_strength()
v_plus = profile.get_v_plus()
v_minus = profile.get_v_minus()

xi = profile.get_xi_coordinate()
v = profile.get_fluid_velocity()
idx_wall = profile.get_wall_index()

shock_index = None
if profile.has_shock():
    shock_index = profile.get_shock_index()

print("solution type :", sol_type.to_string())
print(f"alpha_+       : {alpha_plus:.8f}")
print(f"v_+           : {v_plus:.8f}")
print(f"v_-           : {v_minus:.8f}")
print(f"grid points   : {xi.size}")
print(f"shock present : {profile.has_shock()}")

### Velocity profile

The fluid velocity is shown in the bubble frame.  The duplicate value of $\xi$ at a discontinuity is intentional: it represents the jump at the phase boundary or shock without artificially interpolating through it.

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.4))
ax.plot(xi, v, lw=2.0, label=r"$v(\xi)$")
ax.axvline(xi_w, ls="--", lw=1.2, label=r"$\xi_w$")
ax.axvline(c_plus, ls=":", lw=1.1, label=r"$c_+$")
ax.axvline(c_minus, ls=":", lw=1.1, label=r"$c_-$")
if shock_index is not None:
    ax.axvline(xi[shock_index], ls="--", lw=1.0, label=r"$\xi_{\rm sh}$")
ax.axhline(0.0, lw=0.8)
ax.set_xlim(0.0, 1.0)
ax.set_xlabel(r"$\xi=r/t$")
ax.set_ylabel(r"$v(\xi)$")
ax.grid(True, alpha=0.3)
ax.legend(frameon=False)
fig.tight_layout()

### Enthalpy density

The enthalpy profile is fixed by the velocity solution and the Rankine–Hugoniot matching conditions.  We normalize to the far-front value $w_N$, which removes the arbitrary overall thermal normalization relevant to the stationary hydrodynamics.

In [ ]:
w = profile.get_enthalpy_density()
w_N = w[-1]

fig, ax = plt.subplots(figsize=(7.2, 4.4))
ax.plot(xi, w / w_N, lw=2.0)
ax.axvline(xi_w, ls="--", lw=1.2)
ax.set_xlim(0.0, 1.0)
ax.grid(True, alpha=0.3)
ax.set_xlabel(r"$\xi=r/t$")
ax.set_ylabel(r"$w(\xi)/w_N$")
fig.tight_layout()

print(f"w_+/w_N = {w[idx_wall] / w_N:.8f}")
print(f"w_-/w_N = {w[idx_wall - 1] / w_N:.8f}")

### Energy density

For the CSS equation of state, the thermal contribution satisfies $e_{\rm th}=w/(1+c_s^2)$.  The full energy-density profile additionally contains the phase-dependent vacuum term fixed by the wall matching.

In [ ]:
e = profile.get_energy_density()

fig, ax = plt.subplots(figsize=(7.2, 4.4))
ax.plot(xi, e / w_N, lw=2.0)
ax.axvline(xi_w, ls="--", lw=1.2)
ax.set_xlim(0.0, 1.0)
ax.grid(True, alpha=0.3)
ax.set_xlabel(r"$\xi=r/t$")
ax.set_ylabel(r"$e(\xi)/w_N$")
fig.tight_layout()

### Pressure

The pressure and energy density are reconstructed consistently from the same CSS equation of state.  Their jumps at the wall are physical; the continuous parts evolve with the self-similar flow.

In [ ]:
p = profile.get_pressure()

fig, ax = plt.subplots(figsize=(7.2, 4.4))
ax.plot(xi, p / w_N, lw=2.0)
ax.axvline(xi_w, ls="--", lw=1.2)
ax.set_xlim(0.0, 1.0)
ax.grid(True, alpha=0.3)
ax.set_xlabel(r"$\xi=r/t$")
ax.set_ylabel(r"$p(\xi)/w_N$")
fig.tight_layout()

### Temperature

Velocity and enthalpy are fixed by the hydrodynamic problem, but the relative thermal normalization of the two phases is an additional input.  We parameterize it by

$$
\Psi_N\equiv\frac{w_-(T_N)}{w_+(T_N)}.
$$

For this positive-$\alpha_N$ benchmark we take $\Psi_N=0.91$ as a representative thermodynamically allowed completion and set $T_N=1$.  In a model application, $\Psi_N$ should be obtained from the microscopic equation of state and checked against the wall entropy condition.

In [ ]:
Psi_N = 0.91
T_N = 1.0

T = profile.get_temperature(Psi_N, T_N)

fig, ax = plt.subplots(figsize=(7.2, 4.4))
ax.plot(xi, T / T_N, lw=2.0)
ax.axvline(xi_w, ls="--", lw=1.2)
ax.set_xlim(0.0, 1.0)
ax.grid(True, alpha=0.3)
ax.set_xlabel(r"$\xi=r/t$")
ax.set_ylabel(r"$T(\xi)/T_N$")
fig.tight_layout()

print(f"T_+/T_N = {T[idx_wall] / T_N:.8f}")
print(f"T_-/T_N = {T[idx_wall - 1] / T_N:.8f}")

### Entropy density

The entropy density follows from $s=w/T$.  Entropy production at discontinuities can be checked using the `profile_calculator.check_entropy_flux` function, which returns `None` if the second law of thermodynamics is satisfied and otherwise the index in the profile where it is violated.  A thermodynamic completion is physical only when the entropy flux has the correct orientation across the wall and every shock is compressive.

In [ ]:
s = pc.compute_entropy_density_profile(w, T)
s_N = w_N / T_N
bad_idx = pc.check_entropy_flux(xi, v, s)
if bad_idx is None:
    print("The profile satisfies the entropy flux condition.")
else:
    print(f"The entropy flux condition is violated at xi = {xi[bad_idx]}.")

fig, ax = plt.subplots(figsize=(7.2, 4.4))
ax.plot(xi, s / s_N, lw=2.0)
ax.axvline(xi_w, ls="--", lw=1.2)
ax.set_xlim(0.0, 1.0)
ax.grid(True, alpha=0.3)
ax.set_xlabel(r"$\xi=r/t$")
ax.set_ylabel(r"$s(\xi)/s_N$")
fig.tight_layout()

In [ ]:
# ============================================================
# Hydrodynamic and thermodynamic profiles
# ============================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(10.6, 4.0),
    sharex=True,
)

ax_v, ax_th = axes

# ------------------------------------------------------------
# Left: fluid velocity
# ------------------------------------------------------------
ax_v.plot(
    xi,
    v,
    lw=2.0,
    color="k",
    label=r"$v$",
)

ax_v.axvline(xi_w, color="0.35", ls="--", lw=1.0)

if shock_index is not None:
    ax_v.axvline(
        xi[shock_index],
        color="0.5",
        ls=":",
        lw=1.0,
    )

ax_v.axhline(0.0, color="0.7", lw=0.8)

ax_v.set_xlim(0.0, 1.0)
ax_v.grid(True, alpha=0.3)
ax_v.set_xlabel(r"$\xi=r/t$", fontsize=14)
ax_v.set_ylabel(r"$v(\xi)$", fontsize=14)


# ------------------------------------------------------------
# Right: thermodynamic profiles
# ------------------------------------------------------------
ax_th.plot(xi, w / w_N, lw=1.8, label=r"$w/w_N$")
ax_th.plot(xi, e / w_N, lw=1.8, label=r"$e/w_N$")
ax_th.plot(xi, p / w_N, lw=1.8, label=r"$p/w_N$")
ax_th.plot(xi, T / T_N, lw=1.8, label=r"$T/T_N$")
ax_th.plot(xi, s / s_N, lw=1.8, label=r"$s/s_N$")

ax_th.axvline(xi_w, color="0.35", ls="--", lw=1.0)

if shock_index is not None:
    ax_th.axvline(
        xi[shock_index],
        color="0.5",
        ls=":",
        lw=1.0,
    )

ax_th.axhline(0.0, color="0.7", lw=0.8)

ax_th.set_xlim(0.0, 1.0)
ax_th.grid(True, alpha=0.3)
ax_th.set_xlabel(r"$\xi=r/t$", fontsize=14)
ax_th.set_ylabel(r"normalized thermodynamic profiles", fontsize=14)

ax_th.legend(
    frameon=False,
    ncol=2,
    fontsize=12,
)

fig.tight_layout()
plt.show()

### Kinetic-energy fraction

The kinetic-energy fraction or dimensionless efficiency is calculated from the single-bubble kinetic energy
$$
I_K=\int d\xi\,\xi^2 w(\xi)\gamma^2(\xi)v^2(\xi).
$$
Different normalization conventions are used in the literature.  The default normalization in `HydróDex` is `TotalEnergyDensity`, which provides a natural bounded kinetic-energy fraction used for the sound-shell normalization.  The conventions `VacuumEnergy` and `AlphaPseudoTrace` can be used for comparison with the literature, but need not share the same interpretation away from the conformal bag limit.

In [ ]:
normalizations = {
    "total energy": pc.EfficiencyNormalization.TotalEnergyDensity,
    "vacuum energy": pc.EfficiencyNormalization.VacuumEnergy,
    "pseudo-trace": pc.EfficiencyNormalization.AlphaPseudoTrace,
}

for label, normalization in normalizations.items():
    try:
        value = profile.get_efficiency(normalization)
        print(f"{label:14s}: {value:.8e}")
    except (RuntimeError, ValueError) as exc:
        print(f"{label:14s}: undefined ({exc})")

## 2.3 Scan in wall velocity at fixed $\alpha_N$

At fixed $(\alpha_N,c_+,c_-)$, varying $\xi_w$ traces the allowed stationary branches.  The following scan keeps only successful `FluidProfile` constructions and records the wall-frame velocities $v_\pm$, the local strength $\alpha_+$, and the shock position when present.  This is a convenient first diagnostic before a more expensive observable scan.

In [ ]:
hydrodex.set_noninjectivity_check(hydrodex.CheckMode.SILENT)
hydrodex.set_consistency_check(hydrodex.CheckMode.SILENT)

alpha_N_scan = 0.05
xi_scan = np.linspace(0.01, 0.99, 191)
scan_rows = []

for xw in xi_scan:
    try:
        prof = hydrodex.FluidProfile(
            xi_w=xw,
            alpha_N=alpha_N_scan,
            cp=c_plus,
            cm=c_minus,
            step_size=2.0e-5,
        )
        sol = prof.get_solution_type()
        ap = prof.get_wall_transition_strength()
        vp = prof.get_v_plus()
        vm = prof.get_v_minus()
        xsh = prof.get_shock_position() if prof.has_shock() else np.nan
        scan_rows.append((xw, vp, vm, ap, xsh, sol.to_string()))
    except (RuntimeError, ValueError, OverflowError):
        pass

scan_rows = np.asarray(scan_rows, dtype=object)
print(f"successful points: {len(scan_rows)} / {len(xi_scan)}")

hydrodex.set_noninjectivity_check(hydrodex.CheckMode.WARNING)
hydrodex.set_consistency_check(hydrodex.CheckMode.WARNING)

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7.6, 7.0), sharex=True)

xw = scan_rows[:, 0].astype(float)
vp = scan_rows[:, 1].astype(float)
vm = scan_rows[:, 2].astype(float)
ap = scan_rows[:, 3].astype(float)
xsh = scan_rows[:, 4].astype(float)

ax1.plot(xw, vp, lw=1.8, label=r"$v_+$")
ax1.plot(xw, vm, lw=1.8, label=r"$v_-$")
if np.any(np.isfinite(xsh)):
    ax1.plot(xw, xsh, lw=1.2, label=r"$\xi_{\rm sh}$")
ax1.axhline(c_plus, ls=":", lw=1.0, label=r"$c_+$")
ax1.axhline(c_minus, ls=":", lw=1.0, label=r"$c_-$")
ax1.set_ylabel(r"wall-frame velocity")
ax1.legend(frameon=False, ncol=3)

ax2.plot(xw, ap, lw=1.8)
ax2.axhline(alpha_N_scan, ls=":", lw=1.0, label=r"$\alpha_N$")
ax2.set_xlabel(r"$\xi_w$")
ax2.set_ylabel(r"$\alpha_+(T_+)$")
ax2.legend(frameon=False)

fig.tight_layout()

## 2.4 Hydrodynamic boundaries and the $(\xi_w,\alpha_N)$ phase space

For a fixed pair $(c_+,c_-)$, each `HydroSolutionType` has an allowed interval in $\xi_w$ and corresponding lower and upper bounds in strength.  `xi_min/max` return the kinematic wall-velocity domain, while `alpha_min/max(..., return_alpha_N=True)` return its projected boundaries in the physical far-front variable $\alpha_N$.

The endpoints are excluded by a small numerical cut because several branches become singular or degenerate exactly at their limiting points.  The present benchmark lies outside the fitted non-injective sound-speed region, so the automatic projected boundaries are single-valued.  In a folded regime, branch-specific scans in $\alpha_+$ are required to expose the full image of a non-injective sheet.

In [ ]:
branches = [
    pc.HydroSolutionType.Deflagration,
    pc.HydroSolutionType.Hybrid,
    pc.HydroSolutionType.HybridPlus,
    pc.HydroSolutionType.Detonation,
    pc.HydroSolutionType.InverseDetonation,
    pc.HydroSolutionType.InverseDetonationPlus,
    pc.HydroSolutionType.InverseHybrid,
    pc.HydroSolutionType.InverseDeflagration,
    pc.HydroSolutionType.MixedHybridMinus,
    pc.HydroSolutionType.MixedHybridPlus,
]

branch_labels = {
    pc.HydroSolutionType.Deflagration: r"direct deflagration",
    pc.HydroSolutionType.Hybrid: r"direct hybrid, $v_+<c_-$",
    pc.HydroSolutionType.HybridPlus: r"direct hybrid, $v_+>c_-$",
    pc.HydroSolutionType.Detonation: r"direct detonation",
    pc.HydroSolutionType.InverseDetonation: r"inverse detonation, $-$ sheet",
    pc.HydroSolutionType.InverseDetonationPlus: r"inverse detonation, $+$ sheet",
    pc.HydroSolutionType.InverseHybrid: r"inverse hybrid",
    pc.HydroSolutionType.InverseDeflagration: r"inverse deflagration",
    pc.HydroSolutionType.MixedHybridMinus: r"mixed hybrid, $v_-=c_-$",
    pc.HydroSolutionType.MixedHybridPlus: r"mixed hybrid, $v_+=c_+$",
}

N_boundary = 90
boundary_data = {}

for sol in branches:
    try:
        lo = pc.xi_min(sol, c_plus, c_minus)
        hi = pc.xi_max(sol, c_plus, c_minus)
    except (RuntimeError, ValueError):
        continue

    if not (np.isfinite(lo) and np.isfinite(hi) and lo < hi):
        continue

    cut = 2.0e-4 * max(1.0, hi - lo)
    x_grid = np.linspace(lo + cut, hi - cut, N_boundary)
    a_lo = np.full_like(x_grid, np.nan)
    a_hi = np.full_like(x_grid, np.nan)

    for i, xw_i in enumerate(x_grid):
        try:
            a1 = pc.alpha_min(sol, xw_i, c_plus, c_minus, True)
            a2 = pc.alpha_max(sol, xw_i, c_plus, c_minus, True)
            a_lo[i], a_hi[i] = min(a1, a2), max(a1, a2)
        except (RuntimeError, ValueError, OverflowError):
            pass

    boundary_data[sol] = (x_grid, a_lo, a_hi)

print(f"computed boundaries for {len(boundary_data)} branches")
for sol, (xg, alo, ahi) in boundary_data.items():
    good = np.isfinite(alo) & np.isfinite(ahi)
    print(f"  {branch_labels[sol]:32s}: {np.count_nonzero(good):3d} points")

In [ ]:
# ============================================================
# Hydrodynamic phase space in (xi_w, alpha_N)
# ============================================================

color_dict = {
    pc.HydroSolutionType.Detonation: "#08519c",
    pc.HydroSolutionType.HybridPlus: "#3182bd",
    pc.HydroSolutionType.Hybrid: "#6baed6",
    pc.HydroSolutionType.Deflagration: "#bdd7e7",
    pc.HydroSolutionType.InverseDetonationPlus: "#a50f15",
    pc.HydroSolutionType.InverseDetonation: "#de2d26",
    pc.HydroSolutionType.InverseHybrid: "#fb6a4a",
    pc.HydroSolutionType.InverseDeflagration: "#fcae91",
    pc.HydroSolutionType.MixedHybridPlus: "#756bb1",
    pc.HydroSolutionType.MixedHybridMinus: "#31a354",
}

# Order used only for the legend.
legend_order = [
    pc.HydroSolutionType.Detonation,
    pc.HydroSolutionType.HybridPlus,
    pc.HydroSolutionType.Hybrid,
    pc.HydroSolutionType.Deflagration,
    pc.HydroSolutionType.InverseDetonationPlus,
    pc.HydroSolutionType.InverseDetonation,
    pc.HydroSolutionType.InverseHybrid,
    pc.HydroSolutionType.InverseDeflagration,
    pc.HydroSolutionType.MixedHybridPlus,
    pc.HydroSolutionType.MixedHybridMinus,
]

# ------------------------------------------------------------
# Publication-style figure size
# ------------------------------------------------------------
text_width_pt = 510.0

fig_width = text_width_pt / 72.7
left_margin = 0.75
right_margin = 0.25
top_margin = 0.30
bottom_margin = 1.15

ax_width = fig_width - left_margin - right_margin
ax_height = ax_width

fig_height = ax_height + top_margin + bottom_margin

fig, ax = plt.subplots(
    figsize=(fig_width, fig_height)
)

fig.subplots_adjust(
    left=left_margin / fig_width,
    right=1.0 - right_margin / fig_width,
    bottom=bottom_margin / fig_height,
    top=1.0 - top_margin / fig_height,
)

# ------------------------------------------------------------
# Solution regions
# ------------------------------------------------------------
for sol in legend_order:

    if sol not in boundary_data:
        continue

    xg, alo, ahi = boundary_data[sol]

    good = np.isfinite(alo) & np.isfinite(ahi)

    if not np.any(good):
        continue

    x = xg[good]
    ylo = alo[good]
    yhi = ahi[good]

    color = color_dict[sol]

    ax.fill_between(
        x,
        ylo,
        yhi,
        facecolor=color,
        edgecolor="none",
        rasterized=True,
        zorder=2,
    )

    # Black phase-space boundaries.
    ax.plot(
        x,
        ylo,
        color="k",
        lw=1.0,
        zorder=3,
    )

    ax.plot(
        x,
        yhi,
        color="k",
        lw=1.0,
        zorder=3,
    )

    # Close the region at the numerically sampled endpoints.
    ax.plot(
        [x[0], x[0]],
        [ylo[0], yhi[0]],
        color="k",
        lw=1.0,
        zorder=3,
    )

    ax.plot(
        [x[-1], x[-1]],
        [ylo[-1], yhi[-1]],
        color="k",
        lw=1.0,
        zorder=3,
    )


# ------------------------------------------------------------
# Reference lines
# ------------------------------------------------------------
ax.axhline(
    0.0,
    color="k",
    lw=1.0,
    zorder=4,
)

ax.axvline(
    c_minus,
    color="k",
    ls="--",
    lw=1.0,
    zorder=4,
)

ax.axvline(
    c_plus,
    color="k",
    ls=":",
    lw=1.0,
    zorder=4,
)


# ------------------------------------------------------------
# Axes
# ------------------------------------------------------------
ax.set_xlim(0.0, 1.0)
ax.set_ylim(-1.0, 1.0)

numeric_ticks = np.linspace(0.0, 1.0, 6)

xticks = sorted({
    round(float(t), 10)
    for t in [*numeric_ticks, c_plus, c_minus]
})

xtick_labels = []

for t in xticks:

    if np.isclose(t, c_plus):
        xtick_labels.append(r"$c_+$")

    elif np.isclose(t, c_minus):
        xtick_labels.append(r"$c_-$")

    elif np.isclose(t, 0.0):
        xtick_labels.append(r"$0$")

    elif np.isclose(t, 1.0):
        xtick_labels.append(r"$1$")

    else:
        xtick_labels.append(rf"${t:g}$")

ax.set_xticks(
    xticks,
    labels=xtick_labels,
)

ax.set_xlabel(r"$\xi_w$")
ax.set_ylabel(r"$\alpha_N$")

ax.set_title(
    rf"$c_+={c_plus:g},\qquad c_-={c_minus:g}$"
)


# ------------------------------------------------------------
# Legend
# ------------------------------------------------------------
branch_handles = [
    Patch(
        facecolor=color_dict[sol],
        edgecolor="none",
        label=branch_labels[sol],
    )
    for sol in legend_order
    if sol in boundary_data
]

fig.legend(
    handles=branch_handles,
    loc="lower center",
    bbox_to_anchor=(0.55, -0.04),
    ncols=3,
    frameon=False,
    fontsize=10,
)

plt.show()

## 2.5 Explicit construction from a chosen $\alpha_+$

The high-level constructor takes $\alpha_N$.  For branch-resolved work one can instead choose a specific `HydroSolutionType` and a local wall strength $\alpha_+$.  This is the appropriate low-level interface when the desired stationary completion is already known.

The sequence is

$$
(\text{branch},\xi_w,\alpha_+,c_+,c_-)
\longrightarrow
\alpha_N
\longrightarrow
(v_+,v_-)
\longrightarrow
v(\xi),w(\xi),e(\xi),p(\xi).
$$

In a folded branch, specifying $\alpha_+$ also selects a particular root directly; the current `find_alpha_plus` interface is a single-root inversion and does not enumerate all roots automatically.

In [ ]:
sol_explicit = pc.HydroSolutionType.MixedHybridMinus
xi_w_explicit = 0.45
alpha_plus_explicit = 0.05

# Check that the chosen local strength lies inside the branch interval.
ap_min = pc.alpha_min(sol_explicit, xi_w_explicit, c_plus, c_minus, False)
ap_max = pc.alpha_max(sol_explicit, xi_w_explicit, c_plus, c_minus, False)
ap_lo, ap_hi = min(ap_min, ap_max), max(ap_min, ap_max)

if not (ap_lo < alpha_plus_explicit < ap_hi):
    raise ValueError(
        f"alpha_+={alpha_plus_explicit} is outside the local branch interval "
        f"({ap_lo}, {ap_hi})."
    )

alpha_N_explicit = pc.compute_alpha_N(
    sol_explicit,
    alpha_plus_explicit,
    xi_w_explicit,
    c_plus,
    c_minus,
)

v_plus_explicit, v_minus_explicit = pc.compute_fluid_velocities(
    xi_w_explicit,
    alpha_plus_explicit,
    sol_explicit,
    c_plus,
    c_minus,
)

print("branch       :", sol_explicit.to_string())
print(f"alpha_+      : {alpha_plus_explicit:.8f}")
print(f"alpha_N      : {alpha_N_explicit:.8f}")
print(f"v_+          : {v_plus_explicit:.8f}")
print(f"v_-          : {v_minus_explicit:.8f}")
print(f"local bounds : ({ap_lo:.8f}, {ap_hi:.8f})")

In [ ]:
idx_w_explicit, xi_explicit, v_explicit = pc.compute_velocity_profile(
    sol_explicit,
    xi_w_explicit,
    alpha_plus_explicit,
    c_plus,
    c_minus,
)

idx_sh_explicit = None
try:
    idx_sh_explicit = pc.get_shock_index(sol_explicit, len(xi_explicit))
except RuntimeError:
    pass

w_explicit = pc.compute_enthalpy_density(
    xi_explicit,
    v_explicit,
    idx_w_explicit,
    idx_sh_explicit,
    c_plus,
    c_minus,
)

epsilon_explicit = pc.compute_epsilon(
    w_explicit[-1], alpha_N_explicit, c_plus, c_minus
)

e_explicit = pc.compute_energy_density_profile(
    w_explicit, idx_w_explicit, epsilon_explicit, c_plus, c_minus
)
p_explicit = pc.compute_pressure_profile(
    w_explicit, idx_w_explicit, epsilon_explicit, c_plus, c_minus
)

fig, ax = plt.subplots(figsize=(7.2, 4.4))
ax.plot(xi_explicit, v_explicit, lw=2.0)
ax.axvline(xi_w_explicit, ls="--", lw=1.2)
ax.set_xlim(0.0, 1.0)
ax.grid(True, alpha=0.3)
ax.set_xlabel(r"$\xi=r/t$")
ax.set_ylabel(r"$v(\xi)$")
ax.set_title(r"explicit $\alpha_+$ construction")
fig.tight_layout()

# 3. Gravitational-wave signal

The sound-shell calculation starts from the completed single-bubble profile.  Before overlap, $c_+$ and $c_-$ determine the hydrodynamics.  After overlap, the acoustic propagation speed $c_s$ and the equation-of-state parameter $\omega$ controlling the homogeneous expansion enter the time evolution of the source.  They are conceptually distinct inputs.

For a compact single-fluid CSS example we take

$$
c_s=c_-,\qquad \omega=c_-^2,
$$

and use a constant-$\omega$ background.  A radiation-dominated background can instead be selected with `ExpansionHistory.RADIATION_DOMINATION` and $\omega=1/3$.

## 3.1 Reconstruct the fluid profile for the GW benchmark

We intentionally rebuild the profile rather than reusing the object from Sec. 2.  This makes the gravitational-wave section self-contained once the setup and benchmark inputs have been executed.

In [ ]:
gw_xi_w = 0.65
gw_alpha_N = 0.05

gw_profile = hydrodex.FluidProfile(
    xi_w=gw_xi_w,
    alpha_N=gw_alpha_N,
    cp=c_plus,
    cm=c_minus,
    step_size=1.0e-4,
)

gw_xi = gw_profile.get_xi_coordinate()
gw_v = gw_profile.get_fluid_velocity()

gw_K = gw_profile.get_efficiency(pc.EfficiencyNormalization.TotalEnergyDensity)

print("solution type :", gw_profile.get_solution_type().to_string())
print(f"K             : {gw_K:.8e}")

## 3.2 Numerical precision

The grids for the Fourier transform and integrations are controlled by the following parameters:

- `profile_step_size`: decrease it to resolve narrow hydrodynamic shells and shocks.  Converge the fluid profile *before* tuning Fourier integrals.
- `N_chi` and $(\chi_{\min},\chi_{\max})$: control the profile transform and $A^2(\chi)$.  Increase the range if the Fourier tail is truncated; increase the number of points if oscillatory structure is under-resolved.
- `N_qR`: sampling of the kinetic/velocity spectrum.  The range must contain both the peak and the asymptotic tails relevant to its normalization.
- `N_p`, `N_z`: quadrature resolution of the momentum convolution entering the anisotropic-stress correlator.  These are among the most important settings for converging the GW amplitude and near-peak shape, and are typically the dominant computational cost.
- `N_qR_internal`: internal momentum grid used by the GW integral.  Increase it when interpolation of the kinetic spectrum is visibly affecting the result.
- `N_kR`: only the sampling of the final GW spectrum.  Increasing it makes the plotted curve smoother but does not by itself improve the accuracy of each momentum integral.

The source duration $H_*\Delta\tau$ and the choice of `MomentumIntegration` are physical approximations, not numerical precision parameters.  The values below are suitable for a tutorial run.  For publication results, convergence should be established by varying the relevant grids independently.

In [ ]:
GW_NUMERICS = {
    "H_R": 1.0e-2,
    "H_delta_tau": 1.0e-1,
    "Omega_star": 1.0,
    "chi_min": 1.0e-2,
    "chi_max": 1.0e4,
    "N_chi": 10000,
    "qR_min": 1.0e-2,
    "qR_max": 2.0e4,
    "N_qR": 300,
    "kR_min": 3.0e-2,
    "kR_max": 2.0e3,
    "N_kR": 280,
    "N_qR_internal": 800,
    "N_p": 150,
    "N_z": 64,
}

# A more conservative starting point for final convergence tests is, e.g.,
# N_chi=20000, N_qR=500, N_qR_internal=1200, N_p=180, N_z=72.

acoustic_cs = c_minus
omega = c_minus**2

spec = hydrodex.SoundShellSpectrum(
    profile=gw_profile,
    H_R=GW_NUMERICS["H_R"],
    nhist=ssm.NucleationHistory.EXPONENTIAL,
    exhist=ssm.ExpansionHistory.CONSTANT_EOS,
    cs=acoustic_cs,
    omega=omega,
    Omega_star=GW_NUMERICS["Omega_star"],
)

spec.set_source_duration(GW_NUMERICS["H_delta_tau"])
spec.set_correlator_integration_settings(
    GW_NUMERICS["N_p"],
    GW_NUMERICS["N_z"],
    ssm.MomentumIntegration.FULL,
)

## 3.3 Acoustic Fourier amplitudes: $A^2(\chi)$

The real-space velocity and energy-density perturbations are transformed into acoustic mode amplitudes,

$$
A_\pm(\chi)=-\frac{i}{2}\left[f'(\chi)\pm i c_s\,\ell(\chi)\right],
$$

where $f'$ and $\ell$ are Fourier kernels constructed from the single-bubble profile.  The shape function $A^2(\chi)$ is the first place to diagnose inadequate profile or Fourier resolution: its peak and relevant oscillatory tail should be stable when `profile_step_size`, the $\chi$ range, and `N_chi` are varied.

In [ ]:
chi = np.logspace(
    np.log10(GW_NUMERICS["chi_min"]),
    np.log10(GW_NUMERICS["chi_max"]),
    GW_NUMERICS["N_chi"],
)

fprime, ell = spec.calculate_A_squared(chi)
A2 = spec.get_shape_function()
chi_out = spec.get_chi()

fig, ax = plt.subplots(figsize=(7.2, 4.4))
ax.loglog(chi_out, np.maximum(A2, 1.0e-300), lw=1.8)
ax.set_xlim(1.0e-2, 1.0e4)
ax.grid(True, alpha=0.3)
ax.set_xlabel(r"$\chi$")
ax.set_ylabel(r"$A^2(\chi)$")
fig.tight_layout()

## 3.4 Kinetic and velocity power spectra

The acoustic amplitudes are averaged over the nucleation history to obtain the kinetic spectrum.  We write

$$
E_{\rm kin}(q)=\bar U_f^{\,2}R_*\,\zeta_{\rm kin}(qR_*),
$$

with $\bar U_f^{\,2}$ the mean squared fluid velocity.  `get_velocity_power_spectrum()` returns the corresponding velocity power used to visualize the characteristic shell scales.  The peak and integral normalization should be stable against the $qR_*$ range and sampling.

In [ ]:
qR = np.logspace(
    np.log10(GW_NUMERICS["qR_min"]),
    np.log10(GW_NUMERICS["qR_max"]),
    GW_NUMERICS["N_qR"],
)

spec.calculate_kinetic_spectrum(qR)
qR_out = spec.get_qR()
velocity_power = spec.get_velocity_power_spectrum()
zeta_kin = spec.get_normalized_kinetic_spectrum()

fig, ax = plt.subplots(figsize=(7.2, 4.4))
ax.loglog(qR_out, np.maximum(velocity_power, 1.0e-300), lw=1.8)
ax.set_xlim(1.0e-2, 1.0e4)
ax.grid(True, alpha=0.3)
ax.set_xlabel(r"$qR_*$")
ax.set_ylabel(r"$P_v(qR_*)$")
fig.tight_layout()

print(f"U_f^2     = {spec.get_mean_squared_fluid_velocity():.8e}")
print(f"Omega_K   = {spec.get_kinetic_energy_density_parameter():.8e}")
print(f"qR peak   = {spec.get_kinetic_spectrum_peak_position():.8e}")

## 3.5 Source correlator

The transverse-traceless stress correlator is a convolution of two kinetic spectra.  The two useful diagnostic outputs below separate the shape inherited from the anisotropic stress from the finite-duration time kernel.  If the final GW spectrum is not converged, checking these quantities independently helps distinguish a momentum-convolution issue from a time-evolution issue.

In [ ]:
kR = np.logspace(
    np.log10(GW_NUMERICS["kR_min"]),
    np.log10(GW_NUMERICS["kR_max"]),
    GW_NUMERICS["N_kR"],
)

spec.calculate_gw_spectrum(kR, GW_NUMERICS["N_qR_internal"])

kR_out = spec.get_kR()
K3_zeta_Pi = spec.get_normalized_anisotropic_stress_power_spectrum()
Delta_tilde = spec.get_spectral_modification_function()

fig, ax = plt.subplots(figsize=(7.2, 4.4))
ax.loglog(kR_out, np.maximum(K3_zeta_Pi, 1.0e-300), lw=1.8, label=r"$K^3\zeta_\Pi$")
ax.loglog(kR_out, np.maximum(Delta_tilde, 1.0e-300), lw=1.4, label=r"$\widetilde\Delta$")
ax.set_xlim(3.0e-2, 2.0e3)
ax.grid(True, alpha=0.3)
ax.set_xlabel(r"$kR_*$")
ax.legend(frameon=False)
fig.tight_layout()

## 3.6 Gravitational-wave spectrum at production

The production spectrum is returned as $\Omega_{\rm GW,*}(kR_*)$.  Its overall normalization scales with the square of the kinetic energy density parameter, while its shape retains information about the shell geometry, acoustic propagation, and finite source duration.

In [ ]:
Omega_GW_star = spec.get_gravitational_wave_spectrum()

peak_idx = np.nanargmax(Omega_GW_star)
print(f"(kR_*)_peak       = {kR_out[peak_idx]:.8e}")
print(f"Omega_GW,*^peak   = {Omega_GW_star[peak_idx]:.8e}")

fig, ax = plt.subplots(figsize=(7.2, 4.4))
ax.loglog(kR_out, np.maximum(Omega_GW_star, 1.0e-300), lw=1.9)
ax.set_xlim(3.0e-2, 2.0e3)
ax.grid(True, alpha=0.3)
ax.set_xlabel(r"$kR_*$")
ax.set_ylabel(r"$\Omega_{\rm GW,*}$")
fig.tight_layout()

## 3.7 Redshift to today

For illustration we redshift the production spectrum assuming standard adiabatic expansion after a transition at temperature $T_*$.  Entropy conservation gives the ratio $a_*/a_0$, while the standard radiation transfer factor maps the fractional GW energy density to today.

The helper below returns the physical frequency $f_0$ in Hz and $\Omega_{\rm GW,0}$, **not** $h^2\Omega_{\rm GW,0}$.  The conversion assumes the standard post-transition thermal history; non-standard eras require the corresponding transfer function to be supplied separately.

In [ ]:
def redshift_to_today(
    kR,
    Omega_GW_star,
    H_R,
    T_star_GeV=100.0,
    g_star=100.0,
    g_star_s=None,
):
    if g_star_s is None:
        g_star_s = g_star
    if H_R <= 0.0 or T_star_GeV <= 0.0:
        raise ValueError("H_R and T_star_GeV must be positive.")

    M_pl_GeV = 1.220890e19
    T0_GeV = 2.34865e-13
    g_star_s0 = 3.909
    GeV_to_Hz_angular = 1.519267447e24

    H_star_GeV = 1.66 * np.sqrt(g_star) * T_star_GeV**2 / M_pl_GeV
    a_star_over_a0 = (
        T0_GeV / T_star_GeV * (g_star_s0 / g_star_s)**(1.0 / 3.0)
    )

    frequency_Hz = (
        kR
        / H_R
        * H_star_GeV
        * GeV_to_Hz_angular
        * a_star_over_a0
        / (2.0 * np.pi)
    )

    transfer = 3.57e-5 * (100.0 / g_star)**(1.0 / 3.0)
    return frequency_Hz, transfer * Omega_GW_star


f0, Omega_GW_0 = redshift_to_today(
    kR_out,
    Omega_GW_star,
    GW_NUMERICS["H_R"],
    T_star_GeV=100.0,
    g_star=100.0,
)

peak0 = np.nanargmax(Omega_GW_0)
print(f"f_0^peak          = {f0[peak0]:.8e} Hz")
print(f"Omega_GW,0^peak   = {Omega_GW_0[peak0]:.8e}")

fig, ax = plt.subplots(figsize=(7.2, 4.4))
ax.loglog(f0, np.maximum(Omega_GW_0, 1.0e-300), lw=1.9)
ax.set_xlim(1.0e-5, 1.0e-1)
ax.grid(True, alpha=0.3)
ax.set_xlabel(r"$f_0\;[\mathrm{Hz}]$")
ax.set_ylabel(r"$\Omega_{\rm GW,0}$")
fig.tight_layout()

## 3.8 Small scan across representative hydrodynamic branches

A scan should first be used to establish qualitative trends; only afterwards should the most relevant points be recomputed with the converged single-point settings above. Here we sample a few representative points from the standard direct branches: detonations, deflagrations, and hybrids.

For each hydrodynamic branch, the wall velocity is chosen inside the corresponding physical interval defined by `xi_min` and `xi_max`. At each selected $\xi_w$, the value of $\alpha_N$ is then taken at a fixed fractional position between the branch-specific lower and upper boundaries returned by `alpha_min` and `alpha_max`. This keeps every scan point explicitly inside the intended hydrodynamic region while allowing both $\xi_w$ and $\alpha_N$ to vary.

In [ ]:
# ============================================================
# Small GW scan across representative hydrodynamic branches
# ============================================================

scan_branches = {
    pc.HydroSolutionType.Detonation: {
        "xi_fracs": (0.30, 0.65),
        "alpha_fracs": (0.35, 0.70),
    },
    pc.HydroSolutionType.HybridPlus: {
        "xi_fracs": (0.30, 0.70),
        "alpha_fracs": (0.40, 0.70),
    },
    pc.HydroSolutionType.Hybrid: {
        "xi_fracs": (0.30, 0.70),
        "alpha_fracs": (0.40, 0.70),
    },
    pc.HydroSolutionType.Deflagration: {
        "xi_fracs": (0.30, 0.65),
        "alpha_fracs": (0.35, 0.70),
    },
}

SCAN_NUMERICS = {
    "N_chi": 10000,
    "N_qR": 360,
    "N_kR": 250,
    "N_qR_internal": 320,
    "N_p": 124,
    "N_z": 96,
}

scan_results = []

for branch, cfg in scan_branches.items():

    xi_lo = pc.xi_min(branch, c_plus, c_minus)
    xi_hi = pc.xi_max(branch, c_plus, c_minus)

    if not np.isfinite(xi_lo) or not np.isfinite(xi_hi):
        continue

    if xi_hi <= xi_lo:
        continue

    # Stay safely away from branch endpoints.
    eps_xi = 1.0e-4 * max(1.0, xi_hi - xi_lo)
    xi_lo += eps_xi
    xi_hi -= eps_xi

    for xi_frac, alpha_frac in zip(
        cfg["xi_fracs"],
        cfg["alpha_fracs"],
    ):

        xw_i = xi_lo + xi_frac * (xi_hi - xi_lo)

        try:
            a1 = pc.alpha_min(branch, xw_i, c_plus, c_minus, True)
            a2 = pc.alpha_max(branch, xw_i, c_plus, c_minus, True)

            a_lo = min(a1, a2)
            a_hi = max(a1, a2)

            if not np.isfinite(a_lo) or not np.isfinite(a_hi):
                continue

            aN_i = a_lo + alpha_frac * (a_hi - a_lo)

            # ------------------------------------------------
            # Hydrodynamic profile
            # ------------------------------------------------
            prof_i = hydrodex.FluidProfile(
                xi_w=xw_i,
                alpha_N=aN_i,
                cp=c_plus,
                cm=c_minus,
                step_size=2.0e-4,
            )

            # ------------------------------------------------
            # Sound-shell spectrum
            # ------------------------------------------------
            spec_i = hydrodex.SoundShellSpectrum(
                profile=prof_i,
                H_R=GW_NUMERICS["H_R"],
                nhist=ssm.NucleationHistory.EXPONENTIAL,
                exhist=ssm.ExpansionHistory.CONSTANT_EOS,
                cs=acoustic_cs,
                omega=omega,
                Omega_star=GW_NUMERICS["Omega_star"],
            )

            spec_i.set_source_duration(
                GW_NUMERICS["H_delta_tau"]
            )

            spec_i.set_correlator_integration_settings(
                SCAN_NUMERICS["N_p"],
                SCAN_NUMERICS["N_z"],
                ssm.MomentumIntegration.FULL,
            )

            # ------------------------------------------------
            # Numerical grids
            # ------------------------------------------------
            chi_i = np.logspace(
                -2,
                4,
                SCAN_NUMERICS["N_chi"],
            )

            qR_i = np.logspace(
                -2,
                np.log10(2.0e4),
                SCAN_NUMERICS["N_qR"],
            )

            kR_i = np.logspace(
                np.log10(3.0e-2),
                np.log10(2.0e3),
                SCAN_NUMERICS["N_kR"],
            )

            # ------------------------------------------------
            # GW calculation
            # ------------------------------------------------
            spec_i.calculate_A_squared(chi_i)
            spec_i.calculate_kinetic_spectrum(qR_i)

            spec_i.calculate_gw_spectrum(
                kR_i,
                SCAN_NUMERICS["N_qR_internal"],
            )

            k_i = spec_i.get_kR()

            Om_i = spec_i.get_gravitational_wave_spectrum()

            ip = np.nanargmax(Om_i)

            scan_results.append({
                "branch": branch,
                "xi_w": xw_i,
                "alpha_N": aN_i,
                "solution": prof_i.get_solution_type().to_string(),
                "kR": k_i,
                "Omega": Om_i,
                "k_peak": k_i[ip],
                "Omega_peak": Om_i[ip]
            })

        except (RuntimeError, ValueError, OverflowError) as exc:
            print(
                f"skipping {branch_labels[branch]}, "
                f"xi_w={xw_i:.3f}: {exc}"
            )


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------
n_requested = sum(
    len(cfg["xi_fracs"])
    for cfg in scan_branches.values()
)

print(
    f"successful GW points: "
    f"{len(scan_results)} / {n_requested}"
)

for row in scan_results:
    print(
        f"{branch_labels[row['branch']]:>8s} : "
        f"xi_w={row['xi_w']:.3f}, "
        f"alpha_N={row['alpha_N']:+.5f}, "
        f"{row['solution']}, "
        f"kR_peak={row['k_peak']:.3g}, "
        f"Omega_peak={row['Omega_peak']:.3e}"
    )

In [ ]:
fig, ax = plt.subplots(figsize=(7.6, 4.8))

for row in scan_results:
    ax.loglog(
        row["kR"],
        np.maximum(row["Omega"], 1.0e-300),
        lw=1.5,
        label=(
            rf"$\xi_w={row['xi_w']:.2f},\ "
            rf"\alpha_N={row['alpha_N']:+.3f}$"
        ),
    )

ax.set_xlabel(r"$kR_*$")
ax.set_ylabel(r"$\Omega_{\rm GW,*}$")
ax.set_xlim(3.0e-2, 2.0e2)
ax.set_ylim(1.0e-13, 1.0e-4)
ax.grid(True, alpha=0.3)
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()

## Closing remarks

The intended workflow is therefore:

$$
(c_+,c_-,\xi_w,\alpha_N)
\;\longrightarrow\;
\text{hydrodynamic branch}
\;\longrightarrow\;
\{v,w,e,p,T,s\}(\xi)
\;\longrightarrow\;
A^2
\;\longrightarrow\;
P_v
\;\longrightarrow\;
\Omega_{\rm GW,*}
\;\longrightarrow\;
\Omega_{\rm GW,0}.
$$

For parameter inference or broad scans, convergence should be established hierarchically: first the hydrodynamic profile, then the acoustic transforms and kinetic spectrum, and finally the momentum convolution.  In a sound-speed region flagged as potentially non-injective, the automatic solver should not be interpreted as an all-solutions finder; branch-specific construction in $\alpha_+$ is the appropriate tool for resolving individual stationary completions.